In [158]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [159]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [160]:
data_transformer = transforms.ToTensor()

In [ ]:
embedding_size = 2048
to_decoder_dim = 64
mask_ratio = 0.95
num_attn_heads = 4
num_of_hidden_nodes = 2048
num_of_encoder_blocks = 8
num_of_decoder_blocks = 4
input_size = 224
patch_size = 16
batch_size = 64

loading video and converting to an array of frames with the dimension of (T,C,H,W)

In [162]:
import cv2
import numpy as np

video_path = r'E:\Internships\iit roorke\paper implementatinos\VideoMAE\test_video.mp4'
frames = []  # to store the extracted frames in this list

cap = cv2.VideoCapture(video_path)  #an obect to access the frames

while True:
    ret, frame = cap.read() #reading video, returns ret - whether frame is read correctly or no (True/False) , returns frame

    if not ret or frame is None:
        print("Failed to read frame")
        break    #if ret is false then no frame read, then exit loop

    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) #open cv reads frame in GBR , but we need it in RGB normally

    ini_frame_shape = frame.shape

    frame = cv2.resize(frame, (224,224))

    frames.append(frame)

cap.release()  #releasing all the resources
print("Shape before resizing - ", ini_frame_shape)
print("size of list after resizing - ",len(frames))
video = np.stack(frames)
print("Shape after stacking - ", video.shape)

video = torch.tensor(video, dtype=torch.float32)

#now currently the dim are - (T,H,W,C) but we want (T,C,H,W)
video = video.permute(0,3,1,2)
print("Final shape" , video.shape)


Failed to read frame
Shape before resizing -  (574, 1382, 3)
size of list after resizing -  320
Shape after stacking -  (320, 224, 224, 3)
Final shape torch.Size([320, 3, 224, 224])


In [163]:
#using only 16 frames out of 320 frames so that our memory shouldnt blow up !!! until I reach the final stage of coding
video = video[:16]
#this is just for prototyping purpose

In [164]:
#grouping the 16 frames into groups of 2, so total 8 groups of 2 frames each will be created
#this is being done to reuse the same patching code of ImageMAE
video = video.view(8,2,3,224,224)
print(video.is_contiguous())
print(video.stride())
video = video.reshape(8,6,224,224) #using reshape instead of view, because after view has been used once
#the tensor has become non_contiguous, because view only works on contiguous tensors

#the reason .reshape is used to is, to combine channels and frame groups dim 2x3 =  6 which would
#lead  to such a dim which the F.unfold accepts

'''
N = 8
C = 6
H = 224
W = 224
'''


False
(301056, 150528, 1, 672, 3)


'\nN = 8\nC = 6\nH = 224\nW = 224\n'

In [165]:
patches = F.unfold(video, kernel_size=16, stride=16) #used to create patches
patches = patches.transpose(-1,-2)

print(patches.shape)


# torch.Size([8, 196, 1536]) 
#8 - number of tubelets (temporal)
#196 - total number of patches (224/16 = 14*14 = 196) (spatial)
#1536 - flattened per patch channel array (16*16*6) meaning each patch has 16*16 pixels where each pixel has 6 channels

#another possible explaination
# 8 = temporal tubelet groups
# 196 = spatial patches per group
# 1536 = flattened tubelet

torch.Size([8, 196, 1536])


In [166]:
#embedding these patches to 2048 dimension
embedder = nn.Linear(1536, 2048)
patches = embedder(patches)
print(patches.shape)


torch.Size([8, 196, 2048])


In [167]:
#spatioal (position) + Temporal embedding
#considering spatioal and temporal separately and creating following parameters and then directly adding them 
#nn.Parameter because they are created as parameters so that they can learn how to represent the time and position information in vector
 
#we had 8 tabuletes (each tabulete had 2 frames) so torch.randn(1,8,2048)
#we had 196 frames so torch.randn(1,196,2048)

temp_embed = nn.Parameter(torch.randn(1,8,2048))
pos_embed = nn.Parameter(torch.randn(1,196,2048))
patches = patches + pos_embed

print("Shape of temporal embeddings : ",temp_embed.shape)
print("Shape of position embeddings : ", pos_embed.shape)
print("Shape of patches after adding pos and temp embeddings : ", patches.shape)

patches = patches.view(1,8,196,2048)
print(patches.shape)
# patches = patches.view(1,8*196,2048)

# print("Shape of patches after flattening : ", patches.shape)

Shape of temporal embeddings :  torch.Size([1, 8, 2048])
Shape of position embeddings :  torch.Size([1, 196, 2048])
Shape of patches after adding pos and temp embeddings :  torch.Size([8, 196, 2048])
torch.Size([1, 8, 196, 2048])


In [168]:
num_patches_per_tub = patches.shape[2]

In [169]:
perm_nums = torch.randperm(num_patches_per_tub)
print(perm_nums.shape)    #we will randomly select patches accross 196 patches and then broadcast them to 8 tabulets

torch.Size([196])


In [170]:
mask_idx_count = int(num_patches_per_tub*mask_ratio)
visible_idx_count = num_patches_per_tub - mask_idx_count
print("patches selected for masking per tabulet: ", mask_idx_count)
print("Patches unselected from masking per tabulet: ", visible_idx_count)

patches selected for masking per tabulet:  186
Patches unselected from masking per tabulet:  10


In [171]:
mask_idxs = perm_nums[:mask_idx_count]
visible_idxs = perm_nums[mask_idx_count:]

print(mask_idxs)
print(visible_idxs)

tensor([134, 183,  95,  92, 175, 112, 173, 180,  85,  20,  10, 167, 122,  36,
        124,  47,  59, 182,  84,  64,  39, 187,  32,  31,  57,  55,  37, 125,
        181, 117, 188,  27,  49, 105,  16, 137, 107,  79, 114,  76,  46, 121,
        189,  66,  87,  35, 172, 153, 133, 155,  18,  13, 169,  24,  67,  42,
        154,  48, 126,  53, 165, 185,  25,   6,  56, 127, 164, 103,  44, 131,
         50, 186, 145,  80, 148, 140, 163, 143, 152,  82,  34,  14, 166,  75,
        139,  60, 149, 109, 102,  28,  58, 130, 158,  63,   5, 150, 118, 135,
         93, 176, 190, 170,  65, 119,  40, 159, 115,  45,  73,  74,   8, 123,
         33,  51, 160, 136,  30, 141,   3,  71,  72,  98, 146,  62, 178,  94,
        191,  90,  11,  23,   1,  22, 171, 132, 151, 193,  26, 194,  91,  97,
        120, 156,  86,  96, 113,   9,  78,  12, 110, 177, 116,  19, 179,  21,
        174,  83,  29,  54, 168, 128, 157,  81,  61,  99,  68, 104,   0, 161,
         69,  15, 142, 129,  41, 101, 100,  38,  70,  77, 111,  

In [172]:
masked_patches = patches[:,:, mask_idxs , :]
visible_patches = patches[:,:, visible_idxs, :]

print(masked_patches.shape)
print(visible_patches.shape)

torch.Size([1, 8, 186, 2048])
torch.Size([1, 8, 10, 2048])


In [ ]:
B,T,N,D = visible_patches.shape
visible_patches = visible_patches.reshape(B,T*N,D)

print(visible_patches.shape)

torch.Size([1, 80, 2048])


### ========================================
# Actual Implementation Starts from Here
### ========================================

In [ ]:
class Patchify(nn.Module):
    def __init__(self):
        super().__init__()
     
     #this class actually converts the given input into patches, 
     #kernel = 2, stride = 2 there fore given the 32x32 image, each image will be converted to 32/2 = 16-> 16x16 patches = 256 patches in total 
        

    def forward(self,x):
        patches = F.unfold(x, kernel_size=16, stride=16) #used to create patches
        patches = patches.transpose(-1,-2) #transposing because this returns (B, C, P) (B-Batch, C-flattened CHannels /Dimension, P - patches) but we want (B,P,C)
        return patches


In [ ]:
class Embed_patches_add_pos(nn.Module):
    def __init__(self, embedding_size, no_of_patches):
        super().__init__()

        #This class embeds the patches and then adds position information of the tokens
        #we are not using Embedding class directly from torch because it will lead to look_up tables just like in NLP which we dont require
        #so using a linear layer that would project the existing dimensions (2x2x3 = 12) to desired dimensions
        self.embedding_dim = embedding_size
     
        self.no_of_patches = no_of_patches
        self.embedder = nn.Linear(1536, self.embedding_dim)

        #we are creating position embeddings, 
        #position embeddings are learnable parameters
        #the index is the position, position is not learnt, it is fixed
        #but the vector representation of these positions is learnt  
        self.temp_embed = nn.Parameter(torch.randn(1,8,2048))
        self.pos_embed = nn.Parameter(torch.randn(1,196,2048))
        
        


    def forward(self, x):
        patchx = self.embedder(x)
        patchx = patchx.view(1, 8, 196, self.embedding_dim)

        patchx = patchx + self.pos_embed.unsqueeze(1) + self.temp_embed.unsqueeze(2)

        return patchx



In [ ]:
class Mask(nn.Module):
    def __init__(self, num_patches_per_tub, mask_ratio):
        super().__init__()
        self.num_patches = num_patches_per_tub
        self.mask_ratio = mask_ratio

       

    def forward(self, patches):
        B,T,P,E = patches.shape
        perm_nums = torch.randperm(P*T)   #creating randum permutation of numbers to mask the patches randomly but following uniform distribution
        #perm_nums is inside forward because we want different mask_ids for each tubulet
        mask_idx_count = int(P*T*self.mask_ratio)
        visible_idx_count = P*T - mask_idx_count
        mask_idxs = perm_nums[:mask_idx_count]
        visible_idxs = perm_nums[mask_idx_count:]
        patches = patches.reshape(B, T*P, E)
        masked_patches = patches[:, mask_idxs, :]
        visible_patches = patches[:, visible_idxs, :]
        

        return visible_patches, mask_idxs, perm_nums
        



In [21]:
#MultiHeadAttention
class MultiHeadAttention(nn.Module):
    def __init__(self, embedding_dim,num_patches,num_heads  ):
        super().__init__()
        #definitions
        #input and #output projections
        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.num_patches = num_patches
        self.dim_per_head = self.embedding_dim//self.num_heads

        self.q = nn.Linear(self.embedding_dim, self.embedding_dim)
        self.k = nn.Linear(self.embedding_dim, self.embedding_dim)
        self.v = nn.Linear(self.embedding_dim, self.embedding_dim)
        
        self.proj = nn.Linear(self.embedding_dim, self.embedding_dim)

        #we need to divide the dotproduct with sqroot(dim_per_head) as scalars of dot product
        #increase as vector increases, and that leads to softmax being too peaky, meaning
        #softmax gives maximum attention/value to the bigger values so eventually smaller values are not given much of importance
        #so occurs vanishing gradient problem
        #variance of dot-product grow proportionally to dimension, 
        self.scale =self.dim_per_head**(-0.5)

    def split_heads(self, x):
        batch_sizee , seq_len, emb_dim = x.shape
        x = x.view(batch_sizee, seq_len, self.num_heads, self.dim_per_head )
        return x.permute(0,2,1,3)
    

    def merge_heads(self,x):  
        x = x.permute(0,2,1,3).contiguous()
        batch_sizee , seq_len, num_headss, dim_per_headd = x.shape
        return x.view(batch_sizee, seq_len, self.embedding_dim)

    def forward(self,x):
        Q = self.q(x)
        K = self.k(x)
        V = self.v(x)

        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)

        #K = (64,4,256,32)
        #Q = (64,4,256,32)
        #K.T = (64, 4,32,256)
        #Q@K.T = (64,4, 256, 256)
        self.wei = Q@K.transpose(-1,-2)
        self.wei = self.wei * self.scale
        attn = F.softmax(self.wei, dim = -1)
        self.attn_map = attn
        self.output = attn@V

        self.output = self.merge_heads(self.output)
        return self.proj(self.output)

In [ ]:
class MLP(nn.Module):
    def __init__(self, embedding_dim, num_hidden_nodes, dropout=0.1):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_hidden_nodes = num_hidden_nodes
        self.fc1 = nn.Linear(self.embedding_dim, self.num_hidden_nodes)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(self.num_hidden_nodes, self.embedding_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.fc2(x)

        return x

#MLP is just a part of the Encoder and Decoder architecture to just make the vector representation even more rich by just projecting them into higher dimensions

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, embedding_dim , num_patches, num_attn_heads, num_of_hidden_nodes):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_patches = num_patches
        self.num_attn_heads = num_attn_heads
        self.num_hidden_nodes = num_of_hidden_nodes

        self.norm1 = nn.LayerNorm(self.embedding_dim)
        self.norm2 = nn.LayerNorm(self.embedding_dim)

        self.attention = MultiHeadAttention(self.embedding_dim, self.num_patches, self.num_attn_heads)
        self.mlp = MLP(self.embedding_dim, self.num_hidden_nodes, dropout=0.25)

    def forward(self, x):
        residual1 = x
        x = self.norm1(x)
        attn_opt = self.attention(x)
        attn_opt = attn_opt + residual1

        residual2 = attn_opt
        x = self.norm2(attn_opt)
        mlp_opt = self.mlp(x)
        mlp_opt = mlp_opt+residual2

        return mlp_opt
    
    #Encoder block is simple and follos the same architecture given in "Attention is all you need" paper


In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, to_decoder_dim , num_patches, num_attn_heads, num_of_hidden_nodes):
        super().__init__()
        self.to_decoder_dim = to_decoder_dim
        self.num_patches = num_patches
        self.num_attn_heads = num_attn_heads
        self.num_hidden_nodes = num_of_hidden_nodes

        self.norm1 = nn.LayerNorm(self.to_decoder_dim)
        self.norm2 = nn.LayerNorm(self.to_decoder_dim)

        self.attention = MultiHeadAttention(self.to_decoder_dim, self.num_patches, self.num_attn_heads)
        self.mlp = MLP(self.to_decoder_dim, self.num_hidden_nodes, dropout=0.25)

    def forward(self, x):
        residual1 = x
        x = self.norm1(x)
        attn_opt = self.attention(x)
        attn_opt = attn_opt + residual1

        residual2 = attn_opt
        x = self.norm2(attn_opt)
        mlp_opt = self.mlp(x)
        mlp_opt = mlp_opt+residual2

        return mlp_opt
        
#We are not using cross attention in this decoder like in Attention is all you need, it is not required 
#as according to MAE paper, encoder only sees visible patches , decoder would see both visible and masked patches, so no cross attention which would indirectly inject information into decoder from encoder
#Decoder follows the exact same architecture as of Encoder
#Because the objective of MAE is to keep Decoder light as much as possible

In [ ]:
#This is the main class which integrates each and every step out there

class MAE(nn.Module):
    def __init__(self, batch_size,input_size, patch_size,mask_ratio, embedding_dim , num_attn_heads, num_of_hidden_nodes,num_of_encoder_blocks,num_of_decoder_blocks,to_decoder_dim ):
        super().__init__()
        
        self.batch_size = batch_size
        self.input_size = input_size
        self.patch_size = patch_size
        self.embedding_dim = embedding_dim
        self.num_patches = (self.input_size // self.patch_size)**2
        self.num_attn_heads = num_attn_heads
        self.num_of_hidden_nodes = num_of_hidden_nodes
        self.num_of_encoder_blocks = num_of_encoder_blocks
        self.num_of_decoder_blocks = num_of_decoder_blocks
        self.to_decoder_dim = to_decoder_dim
        self.mask_ratio = mask_ratio
        self.num_of_masked = int(self.num_patches*self.mask_ratio)

        self.layer_to_dec_dim = nn.Linear(self.embedding_dim, self.to_decoder_dim)
        self.mask_tokens = nn.Parameter(torch.randn(1, 1, self.to_decoder_dim))
        self.to_original_dim = nn.Linear(self.to_decoder_dim, 1536)



        self.patching = Patchify()
        self.emb_posemb = Embed_patches_add_pos(self.embedding_dim,self.num_patches)
        self.masking = Mask(self.num_patches, self.mask_ratio)

        self.encoder = nn.Sequential(*[
            EncoderBlock(self.embedding_dim, self.num_patches, self.num_attn_heads, self.num_of_hidden_nodes)
            for _ in range(self.num_of_encoder_blocks)
        ])   #the following has been done, inorder to repeat the encoder self.num_of_encoder_blocks time, so that model can understand representations and symantics better
        #these are not run parallelly, but run sequentially one after the another,so when x goes in and latent comes out, the latent is again fed into encoder in that particular epoch
        
        self.decoder = nn.Sequential(*[
            DecoderBlock(self.to_decoder_dim, self.num_patches, self.num_attn_heads, self.num_of_hidden_nodes)
            for _ in range(self.num_of_decoder_blocks)  
        ])
        #the above description is applicable for decoder as well

        #videomae requirements
        #defining number of tubelents
         


    def forward(self,x):
        raw_patches = self.patching(x)   #creating patches
        patches = self.emb_posemb(raw_patches) #embedding the patches and adding position embeddings
        visible_patches , mask_idxs , perm_nums = self.masking(patches) #masking the patches, which returns visible patches for encoder, ids of masked patches required to collect these later
                                                                        #per_nums for applying inverted permutation inorder to restore the order of masked and unmasked patches collectively
        mask_idxs = mask_idxs.long()  #indexes needs to be in long format to get processed (It was a bug, bcz initially it was in int )

        latent = self.encoder(visible_patches) #output from encoder
        latent = self.layer_to_dec_dim(latent) #reducing the dimension inorder to send inot decoder

        # print("SHape of latent" , latent.shape)  #print statements were used for debugging purpose
        # print("Shape of mask_tokens" , mask_tokens.shape)

        B = latent.shape[0]  #Note: A lesson I learnt!! dont hardcode batchsize in your pipeline, make sure to keep it dynamic as much , wherever as possible
        mask_tokens = self.mask_tokens.repeat(B, self.num_of_masked*8  , 1) #created the mask_tokens with (1,1,dim) but we need it (64,num_of_masked, dim) so using repeat to repeat the same masked tokens accross the batches to keep them consistent
        all_tokens =  torch.cat([latent, mask_tokens], dim=1) #concatinate the masked and visible tokens to send them into decoder
        restored = all_tokens[:,torch.argsort(perm_nums), :] #restore the random order before sending them to into decoder


        decoder_opt = self.decoder(restored)   #output from decoder
        pred_pixels = self.to_original_dim(decoder_opt) #no we need to convert the dim to 12 (original) to compare and calculate the losses

        #expectation of Fold : (B, C*k*k, L)
        #so transpose as our original shape of pred_pixels is : (B, L , C*K*K)
        pred_masked = pred_pixels[:, mask_idxs]
        pred_pixels = pred_pixels.transpose(-1,-2)
        reconstructed = F.fold(
                        pred_pixels,
                        output_size=(224,224),
                        kernel_size=16,
                        stride=16
                    )

        # print(mask_idxs.min())
        # print(mask_idxs.max())
        # print(raw_patches.shape)
        # print(mask_idxs.shape)

        # print(mask_idxs.device)
        # print(raw_patches.device)

        # print(mask_idxs.dtype)

        target_masked = raw_patches[:, mask_idxs]

        loss = F.mse_loss(pred_masked, target_masked) #MSE loss is being used
   
        return reconstructed, loss





        

        
